# 10 — LLM Integration & Async

32 examples covering OpenAI ChatCompletion, Anthropic Claude, AsyncGuard,
streaming, function calling, structured outputs, retry/backoff, FastAPI,
token budgets, multi-turn conversations, and production patterns.

**Installation:**
```bash
pip install guardrails-ai openai anthropic python-dotenv aiohttp litellm
guardrails hub install hub://guardrails/toxic_language
guardrails hub install hub://guardrails/detect_pii
guardrails hub install hub://guardrails/valid_json
guardrails hub install hub://guardrails/profanity_free
```

In [ ]:
import os, asyncio, time, json, logging
from dotenv import load_dotenv
load_dotenv('../.env')

import openai
import anthropic
from guardrails import Guard, AsyncGuard, OnFailAction
from guardrails.errors import ValidationError

oai = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
ant = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
MODEL_OAI = 'gpt-4o-mini'
MODEL_ANT = 'claude-haiku-4-5-20251001'
print('Setup complete.')

In [ ]:
!guardrails hub install hub://guardrails/toxic_language --quiet
!guardrails hub install hub://guardrails/detect_pii --quiet
!guardrails hub install hub://guardrails/valid_json --quiet
!guardrails hub install hub://guardrails/profanity_free --quiet

## Examples 01–04: OpenAI and Anthropic Integration

In [ ]:
# Example 01: OpenAI ChatCompletion integration — guard wraps the LLM call
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=20, max=1000, on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt='What is photosynthesis?',
    model=MODEL_OAI
)
print('OpenAI integration result:', outcome.validated_output[:100] if outcome.validated_output else 'None')

In [ ]:
# Example 02: Explicit model parameter in guard call
from guardrails.hub import ToxicLanguage
guard = Guard().use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
outcome = guard(
    oai.chat.completions.create,
    prompt='Write a professional email declining a meeting request.',
    model='gpt-4o-mini',  # explicit model
    temperature=0.7,
    max_tokens=300
)
print('with explicit model params:', outcome.validated_output[:100] if outcome.validated_output else 'None')

In [ ]:
# Example 03: OpenAI messages API format (system + user message)
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=10, max=2000, on_fail=OnFailAction.NOOP))

# Using messages format directly
resp = oai.chat.completions.create(
    model=MODEL_OAI,
    messages=[
        {'role': 'system', 'content': 'You are a helpful programming assistant.'},
        {'role': 'user', 'content': 'What is a Python decorator?'}
    ]
)
outcome = guard.validate(resp.choices[0].message.content)
print('messages format result:', outcome.validated_output[:100] if outcome.validated_output else 'None')

In [ ]:
# Example 04: Anthropic Claude integration
from guardrails.hub import ToxicLanguage

# Call Anthropic directly, then validate the response
message = ant.messages.create(
    model=MODEL_ANT,
    max_tokens=300,
    messages=[{'role': 'user', 'content': 'Explain what an API is in simple terms.'}]
)
anthropic_text = message.content[0].text

guard = Guard().use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate(anthropic_text)
print('Anthropic + Guard result:', outcome.validated_output[:100] if outcome.validated_output else 'None')

## Examples 05–08: AsyncGuard

In [ ]:
# Example 05: AsyncGuard creation — difference from synchronous Guard
from guardrails.hub import ValidLength
sync_guard = Guard().use(ValidLength(min=10, max=500, on_fail=OnFailAction.EXCEPTION))
async_guard = AsyncGuard().use(ValidLength(min=10, max=500, on_fail=OnFailAction.EXCEPTION))
print('sync Guard  :', type(sync_guard).__name__)
print('async Guard :', type(async_guard).__name__)

In [ ]:
# Example 06: await guard.async_validate() — basic async validation
from guardrails.hub import ValidLength

async def validate_async():
    guard = AsyncGuard().use(ValidLength(min=10, max=500, on_fail=OnFailAction.EXCEPTION))
    outcome = await guard.async_validate('This is an asynchronously validated string.')
    print('async_validate result:', outcome.validation_passed)
    return outcome

asyncio.run(validate_async())

In [ ]:
# Example 07: await guard() with async OpenAI client
from guardrails.hub import ToxicLanguage

async def async_llm_call():
    async_oai = openai.AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    guard = AsyncGuard().use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
    outcome = await guard(
        async_oai.chat.completions.create,
        prompt='What are the benefits of async programming in Python?',
        model=MODEL_OAI
    )
    return outcome.validated_output

result = asyncio.run(async_llm_call())
print('async LLM result:', result[:100] if result else 'None')

In [ ]:
# Example 08: asyncio.gather() — parallel validation of multiple outputs
from guardrails.hub import ValidLength

async def parallel_validation():
    guard = AsyncGuard().use(ValidLength(min=5, max=500, on_fail=OnFailAction.NOOP))
    texts = [
        'First parallel response about machine learning.',
        'Second parallel response about deep learning.',
        'Third parallel response about natural language processing.',
    ]
    results = await asyncio.gather(*[guard.async_validate(t) for t in texts])
    return [(r.validation_passed, r.validated_output[:40]) for r in results]

parallel_results = asyncio.run(parallel_validation())
for passed, preview in parallel_results:
    print(f'  passed={passed}: {preview}')

## Examples 09–11: Streaming

In [ ]:
# Example 09: Streaming with Guard — iterate over validation chunks
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=10, max=3000, on_fail=OnFailAction.NOOP))
fragment_generator = guard(
    oai.chat.completions.create,
    prompt='Explain the concept of recursion in programming.',
    model=MODEL_OAI,
    stream=True
)
full_output = ''
for chunk in fragment_generator:
    if chunk.validated_output:
        full_output = chunk.validated_output
print('streaming complete. Total length:', len(full_output))
print('preview:', full_output[:100])

In [ ]:
# Example 10: Streaming + ToxicLanguage — real-time safety during streaming
from guardrails.hub import ToxicLanguage
guard = Guard().use(ToxicLanguage(on_fail=OnFailAction.NOOP))
fragment_gen = guard(
    oai.chat.completions.create,
    prompt='Explain best practices for API design.',
    model=MODEL_OAI,
    stream=True
)
chunk_count = 0
for chunk in fragment_gen:
    chunk_count += 1
print(f'Processed {chunk_count} stream chunks. Final validation_passed:', chunk.validation_passed)

In [ ]:
# Example 11: Streaming partial output — access final validated_output after stream ends
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=10, max=5000, on_fail=OnFailAction.NOOP))
gen = guard(
    oai.chat.completions.create,
    prompt='List 5 benefits of containerization with Docker.',
    model=MODEL_OAI,
    stream=True
)
last_chunk = None
for last_chunk in gen:
    pass  # consume stream
if last_chunk:
    print('final stream output length:', len(last_chunk.validated_output or ''))
    print('validation passed:', last_chunk.validation_passed)

## Examples 12–14: Function Calling, Structured Outputs, Error Recovery

In [ ]:
# Example 12: OpenAI function calling — guard validates function output
from pydantic import BaseModel

class WeatherQuery(BaseModel):
    city: str
    unit: str

tools = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get weather for a city',
        'parameters': {
            'type': 'object',
            'properties': {
                'city': {'type': 'string'},
                'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']}
            },
            'required': ['city', 'unit']
        }
    }
}]
resp = oai.chat.completions.create(
    model=MODEL_OAI,
    messages=[{'role': 'user', 'content': 'What is the weather in London in celsius?'}],
    tools=tools,
    tool_choice='auto'
)
tool_call = resp.choices[0].message.tool_calls
if tool_call:
    args = json.loads(tool_call[0].function.arguments)
    guard = Guard.for_pydantic(output_class=WeatherQuery)
    outcome = guard.validate(json.dumps(args))
    print('function call validated:', outcome.validated_output)

In [ ]:
# Example 13: OpenAI structured outputs mode + Guard
from pydantic import BaseModel
from typing import Literal

class TopicClassification(BaseModel):
    topic: Literal['tech', 'finance', 'health', 'sports']
    confidence: float

guard = Guard.for_pydantic(output_class=TopicClassification)
outcome = guard(
    oai.chat.completions.create,
    prompt='Classify: "Stock markets hit record high today". Return JSON with topic and confidence.',
    model=MODEL_OAI,
    response_format={'type': 'json_object'}
)
print('structured output classification:', outcome.validated_output)

In [ ]:
# Example 14: Error recovery — catch ValidationError, retry with fallback prompt
from guardrails.hub import ValidLength

def call_with_fallback(primary_prompt: str, fallback_prompt: str) -> str:
    guard = Guard().use(ValidLength(min=50, max=1000, on_fail=OnFailAction.EXCEPTION))
    try:
        outcome = guard(
            oai.chat.completions.create,
            prompt=primary_prompt,
            model=MODEL_OAI
        )
        return outcome.validated_output
    except ValidationError:
        # Fallback: retry with more detailed prompt
        resp = oai.chat.completions.create(
            model=MODEL_OAI,
            messages=[{'role': 'user', 'content': fallback_prompt}]
        )
        return resp.choices[0].message.content

result = call_with_fallback(
    'Explain APIs.',
    'Please write at least 3 sentences explaining what APIs are and how they work.'
)
print('fallback result:', result[:100])

## Examples 15–17: Custom LLMs, Anthropic Streaming, Retry

In [ ]:
# Example 15: Custom LLM callable — wrapping any model (Ollama / local) in guardrails
from guardrails.hub import ValidLength

def my_custom_llm(prompt: str, **kwargs) -> str:
    # Replace this with your actual LLM call (Ollama, vLLM, etc.)
    # Simulated response for demonstration
    return f'This is a simulated response from a custom LLM for: {prompt[:30]}'

guard = Guard().use(ValidLength(min=10, max=500, on_fail=OnFailAction.NOOP))

# Validate output from custom LLM
custom_output = my_custom_llm('What is Docker?')
outcome = guard.validate(custom_output)
print('custom LLM output validated:', outcome.validation_passed)
print('output:', outcome.validated_output[:80])

In [ ]:
# Example 16: Anthropic streaming — collect and validate streamed response
from guardrails.hub import ToxicLanguage

full_response = ''
with ant.messages.stream(
    model=MODEL_ANT,
    max_tokens=200,
    messages=[{'role': 'user', 'content': 'Describe microservices architecture briefly.'}]
) as stream:
    for text in stream.text_stream:
        full_response += text

# Validate the complete streamed response
guard = Guard().use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate(full_response)
print('Anthropic streaming + guard:', outcome.validation_passed)
print('response length:', len(full_response))

In [ ]:
# Example 17: Retry with exponential backoff for rate-limited calls
from guardrails.hub import ValidLength

def guarded_call_with_backoff(prompt: str, max_retries: int = 3) -> str:
    guard = Guard().use(ValidLength(min=20, max=1000, on_fail=OnFailAction.EXCEPTION))
    for attempt in range(max_retries):
        try:
            outcome = guard(
                oai.chat.completions.create,
                prompt=prompt,
                model=MODEL_OAI
            )
            return outcome.validated_output
        except openai.RateLimitError:
            wait = 2 ** attempt
            print(f'Rate limited. Waiting {wait}s before retry {attempt+1}...')
            time.sleep(wait)
        except ValidationError as e:
            print(f'Validation failed on attempt {attempt+1}: {e}')
            break
    return 'Max retries exceeded'

result = guarded_call_with_backoff('Explain what cloud computing is.')
print('backoff result:', result[:100])

## Examples 18–20: Multi-turn, FastAPI, System Prompts

In [ ]:
# Example 18: Multi-turn conversation — validate each user input and assistant response
from guardrails.hub import ToxicLanguage, ValidLength

conversation_history = []
input_guard = Guard().use(ValidLength(min=5, max=500, on_fail=OnFailAction.EXCEPTION))
output_guard = Guard().use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))

def chat_turn(user_message: str) -> str:
    input_guard.validate(user_message)  # validate input
    conversation_history.append({'role': 'user', 'content': user_message})
    resp = oai.chat.completions.create(model=MODEL_OAI, messages=conversation_history)
    assistant_msg = resp.choices[0].message.content
    output_guard.validate(assistant_msg)  # validate output
    conversation_history.append({'role': 'assistant', 'content': assistant_msg})
    return assistant_msg

r1 = chat_turn('What is a database?')
r2 = chat_turn('What is the difference between SQL and NoSQL?')
print('Turn 1:', r1[:80])
print('Turn 2:', r2[:80])
print('History length:', len(conversation_history), 'messages')

In [ ]:
# Example 19: Guard inside FastAPI endpoint (simulation without running server)
from guardrails.hub import ToxicLanguage, ValidLength

# Simulates what a FastAPI async endpoint would do
async def chat_endpoint(user_message: str) -> dict:
    """Simulated FastAPI POST /chat endpoint with guardrails."""
    async_oai = openai.AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    guard = (
        AsyncGuard()
        .use(ValidLength(min=5, max=500, on_fail=OnFailAction.EXCEPTION))
        .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
    )
    try:
        # Validate input
        await guard.async_validate(user_message)
        # Generate response
        outcome = await guard(
            async_oai.chat.completions.create,
            prompt=user_message,
            model=MODEL_OAI
        )
        return {'status': 'ok', 'response': outcome.validated_output}
    except ValidationError as e:
        return {'status': 'error', 'detail': str(e)[:100]}

result = asyncio.run(chat_endpoint('Explain what REST APIs are.'))
print('FastAPI endpoint result:', result['status'])
print('response:', result.get('response', result.get('detail', ''))[:100])

In [ ]:
# Example 20: System prompt injection for REASK — guard modifies system prompt
from guardrails.hub import ValidLength

guard = Guard().use(ValidLength(min=100, max=2000, on_fail=OnFailAction.REASK))

# The guard will auto-inject reask instructions into the prompt on failure
outcome = guard(
    oai.chat.completions.create,
    messages=[
        {'role': 'system', 'content': 'You are a concise assistant. Provide detailed technical answers.'},
        {'role': 'user', 'content': 'What is Kubernetes?'}
    ],
    model=MODEL_OAI,
    num_reasks=1
)
print('system prompt + guard result:', outcome.validation_passed)
print('output length:', len(outcome.validated_output or ''))

## Examples 21–24: Temperature, Token Budget, Batch, Timeout

In [ ]:
# Example 21: Temperature control for REASK — lower temp on retry for deterministic fix
from guardrails.hub import ValidLength

guard = Guard().use(ValidLength(min=80, max=500, on_fail=OnFailAction.REASK))

# First call with high temperature; if REASK needed, use lower temperature
outcome = guard(
    oai.chat.completions.create,
    prompt='Describe TCP/IP in detail.',
    model=MODEL_OAI,
    temperature=0.3,  # lower temperature for more consistent outputs
    num_reasks=1
)
print('temperature-controlled result:', outcome.validation_passed)
print('output length:', len(outcome.validated_output or ''))

In [ ]:
# Example 22: Token budget awareness — check tokens consumed after call
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=10, max=1000, on_fail=OnFailAction.NOOP))
guard(
    oai.chat.completions.create,
    prompt='What are the SOLID principles in software engineering?',
    model=MODEL_OAI
)
if guard.history:
    call = guard.history[0]
    print('tokens consumed:', call.tokens_consumed)
    print('prompt tokens  :', call.prompt_tokens_consumed)
    print('completion toks:', call.completion_tokens_consumed)

In [ ]:
# Example 23: Batch processing with AsyncGuard — parallel validation
from guardrails.hub import ToxicLanguage

async def batch_validate(texts):
    guard = AsyncGuard().use(ToxicLanguage(on_fail=OnFailAction.NOOP))
    tasks = [guard.async_validate(t) for t in texts]
    outcomes = await asyncio.gather(*tasks, return_exceptions=True)
    return [
        o.validation_passed if not isinstance(o, Exception) else False
        for o in outcomes
    ]

batch = [
    'A professional response about software.',
    'Another informative message about engineering.',
    'A helpful explanation of design patterns.',
    'A clear description of cloud services.',
]
results = asyncio.run(batch_validate(batch))
print(f'Batch of {len(batch)}: {sum(results)} passed, {len(results)-sum(results)} failed')

In [ ]:
# Example 24: Timeout handling for async guard calls
from guardrails.hub import ValidLength

async def guarded_call_with_timeout(prompt: str, timeout_secs: float = 30.0):
    async_oai = openai.AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    guard = AsyncGuard().use(ValidLength(min=10, max=1000, on_fail=OnFailAction.NOOP))
    try:
        outcome = await asyncio.wait_for(
            guard(
                async_oai.chat.completions.create,
                prompt=prompt,
                model=MODEL_OAI
            ),
            timeout=timeout_secs
        )
        return outcome.validated_output
    except asyncio.TimeoutError:
        print(f'Timeout after {timeout_secs}s')
        return None

result = asyncio.run(guarded_call_with_timeout('What is event-driven architecture?', timeout_secs=30))
print('timeout-guarded result:', result[:100] if result else 'timed out')

## Examples 25–32: Provider Switching, LiteLLM, Metrics, Production

In [ ]:
# Example 25: LLM provider switching — same guard used with OpenAI then manually with Anthropic
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=20, max=1000, on_fail=OnFailAction.NOOP))

# OpenAI
oai_resp = oai.chat.completions.create(
    model=MODEL_OAI,
    messages=[{'role': 'user', 'content': 'Explain caching in 2 sentences.'}]
)
oai_outcome = guard.validate(oai_resp.choices[0].message.content)

# Anthropic
ant_msg = ant.messages.create(
    model=MODEL_ANT, max_tokens=150,
    messages=[{'role': 'user', 'content': 'Explain caching in 2 sentences.'}]
)
ant_outcome = guard.validate(ant_msg.content[0].text)

print('OpenAI result  :', oai_outcome.validation_passed)
print('Anthropic result:', ant_outcome.validation_passed)

In [ ]:
# Example 26: Guard with litellm universal adapter
try:
    import litellm
    LITELLM_AVAILABLE = True
except ImportError:
    LITELLM_AVAILABLE = False
    print('litellm not installed — install with: pip install litellm')

if LITELLM_AVAILABLE:
    from guardrails.hub import ValidLength
    guard = Guard().use(ValidLength(min=10, max=1000, on_fail=OnFailAction.NOOP))

    # litellm provides unified interface across providers
    resp = litellm.completion(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': 'What is load balancing?'}]
    )
    outcome = guard.validate(resp.choices[0].message.content)
    print('litellm + guard:', outcome.validation_passed)

In [ ]:
# Example 27: Input guard pre-LLM — validate user input, short-circuit if unsafe
from guardrails.hub import DetectPII, ValidLength

def safe_pipeline(user_input: str) -> dict:
    input_guard = (
        Guard()
        .use(ValidLength(min=5, max=500, on_fail=OnFailAction.EXCEPTION))
        .use(DetectPII(on_fail=OnFailAction.EXCEPTION))
    )
    try:
        input_guard.validate(user_input)
    except ValidationError as e:
        return {'blocked': True, 'reason': 'input_validation', 'detail': str(e)[:80]}

    resp = oai.chat.completions.create(
        model=MODEL_OAI,
        messages=[{'role': 'user', 'content': user_input}]
    )
    return {'blocked': False, 'response': resp.choices[0].message.content[:100]}

# Clean input
r1 = safe_pipeline('What is TLS encryption?')
print('clean input:', r1)

# Input with PII
r2 = safe_pipeline('My SSN is 123-45-6789, help me with taxes.')
print('PII input:', r2)

In [ ]:
# Example 28: Output guard post-LLM — validate response, REASK if needed
from guardrails.hub import ToxicLanguage, ValidLength
guard = (
    Guard()
    .use(ToxicLanguage(on_fail=OnFailAction.REASK))
    .use(ValidLength(min=30, max=2000, on_fail=OnFailAction.REASK))
)
outcome = guard(
    oai.chat.completions.create,
    prompt='Describe the benefits of unit testing in software development.',
    model=MODEL_OAI,
    num_reasks=1
)
print('output guard with REASK:', outcome.validation_passed)
print('iterations:', len(guard.history[0].iterations) if guard.history else 0)

In [ ]:
# Example 29: Metrics collection — extract latency and token counts
from guardrails.hub import ValidLength

guard = Guard().use(ValidLength(min=10, max=1000, on_fail=OnFailAction.NOOP))
start_time = time.perf_counter()
guard(
    oai.chat.completions.create,
    prompt='What is containerization?',
    model=MODEL_OAI
)
elapsed = time.perf_counter() - start_time

metrics = {
    'latency_ms': round(elapsed * 1000, 2),
    'tokens_consumed': guard.history[0].tokens_consumed if guard.history else None,
    'validation_passed': guard.history[0].status if guard.history else None,
    'iterations': len(guard.history[0].iterations) if guard.history else 0,
}
print('Metrics:', metrics)

In [ ]:
# Example 30: Guard-per-request vs guard reuse — production pattern comparison
from guardrails.hub import ToxicLanguage

# Pattern A: Create new guard per request (stateless, safe for concurrency)
def pattern_a(prompt: str) -> str:
    guard = Guard().use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))  # new each time
    outcome = guard(
        oai.chat.completions.create,
        prompt=prompt,
        model=MODEL_OAI
    )
    return outcome.validated_output

# Pattern B: Reuse guard across requests (faster init, but history accumulates)
SHARED_GUARD = Guard().use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
def pattern_b(prompt: str) -> str:
    outcome = SHARED_GUARD(
        oai.chat.completions.create,
        prompt=prompt,
        model=MODEL_OAI
    )
    return outcome.validated_output

r_a = pattern_a('What is a firewall?')
r_b = pattern_b('What is a VPN?')
print('Pattern A (per-request):', r_a[:60] if r_a else 'None')
print('Pattern B (shared):', r_b[:60] if r_b else 'None')
print('Shared guard history length:', len(SHARED_GUARD.history))

In [ ]:
# Example 31: Guard logging integration — attach Python logging to guard events
import logging
from guardrails.hub import ValidLength

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger('guardrails.example')

guard = Guard().use(ValidLength(min=10, max=500, on_fail=OnFailAction.NOOP))
texts = ['Short', 'A longer response that meets the minimum length requirement for this validator.']
for text in texts:
    outcome = guard.validate(text)
    if outcome.validation_passed:
        logger.info('PASS: %d chars', len(text))
    else:
        logger.warning('FAIL: %d chars — %s', len(text), text[:30])

In [ ]:
# Example 32: AsyncGuard with aiohttp external validator call
import aiohttp
from guardrails.validator_base import Validator, register_validator, PassResult, FailResult

@register_validator(name='async-url-check', data_type='string')
class AsyncURLCheck(Validator):
    async def validate(self, value: str, metadata: dict) -> PassResult | FailResult:
        # Async HTTP request to an external content moderation service
        # Using httpbin.org as a safe test endpoint
        try:
            async with aiohttp.ClientSession() as session:
                async with session.get('https://httpbin.org/status/200', timeout=aiohttp.ClientTimeout(total=5)) as resp:
                    if resp.status == 200:
                        return PassResult()  # external service approved
                    return FailResult(error_message=f'External check failed: {resp.status}')
        except Exception as e:
            return FailResult(error_message=f'External service unavailable: {e}')

async def run_aiohttp_guard():
    guard = AsyncGuard().use(AsyncURLCheck(on_fail=OnFailAction.NOOP))
    outcome = await guard.async_validate('Content to externally validate.')
    print('aiohttp external validator result:', outcome.validation_passed)

asyncio.run(run_aiohttp_guard())